# 03_Futures_Database 数据读取 Demo

本 Notebook 演示 `futures_lake/silver` 中的八张表。公共初始化位于下一单元格；此后每个代码单元格只读取并展示一张表。

请使用项目标准 `latitude` Conda 环境运行。所有读取显式使用 `config/data_contracts.py` 的 Arrow Schema；大表先执行 Hive 分区过滤。

In [ ]:
from __future__ import annotations

import importlib
import pathlib
import sys
from datetime import date

import pyarrow as pa
import pyarrow.dataset as ds
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

PROJECT_ROOT = candidate_root
LAKE_ROOT = PROJECT_ROOT / "03_Futures_Database" / "futures_lake" / "silver"
lakehouse = importlib.import_module("02_Quant_Trading.a01_Data_Collection.c00_lakehouse")

from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    FUTURES_DAILY_SCHEMA,
    FUTURES_FETCH_STATUS_SCHEMA,
    FUTURES_MINUTE_SCHEMA,
    FUTURES_MISSING_BAR_SCHEMA,
    FUTURES_SESSION_SCHEDULE_SIGNAL_SCHEMA,
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    TRADE_CALENDAR_SCHEMA,
)

YEAR_PARTITIONING = lakehouse.hive_partitioning([pa.field("year", pa.int16())])
EXCHANGE_MONTH_PARTITIONING = lakehouse.hive_partitioning([pa.field("exchange_code", pa.string()), pa.field("year", pa.int16()), pa.field("month", pa.int8())])
STATUS_PARTITIONING = lakehouse.hive_partitioning([pa.field("bar_frequency", pa.string()), pa.field("exchange_code", pa.string()), pa.field("year", pa.int16()), pa.field("month", pa.int8())])
MISSING_PARTITIONING = lakehouse.hive_partitioning([pa.field("bar_frequency", pa.string()), pa.field("exchange_code", pa.string()), pa.field("underlying_code", pa.string()), pa.field("year", pa.int16()), pa.field("month", pa.int8())])
MINUTE_PARTITIONING = lakehouse.hive_partitioning([pa.field("exchange_code", pa.string()), pa.field("underlying_code", pa.string()), pa.field("year", pa.int16()), pa.field("month", pa.int8())])

def show_dataframe(dataframe, row_count: int = 10) -> None:
    print(f"rows={len(dataframe):,}, columns={len(dataframe.columns)}")
    display(dataframe.head(row_count))
    display(dataframe.dtypes.rename("dtype"))

print(f"project_root={PROJECT_ROOT}")
print(f"lake_root={LAKE_ROOT}")

## 1. `dim_trade_calendar`：交易日历

In [ ]:
trade_calendar_df = lakehouse.read_dataset(
    LAKE_ROOT / "dim_trade_calendar", YEAR_PARTITIONING, TRADE_CALENDAR_SCHEMA,
    ["calendar_date", "date_key", "is_trading_day", "weekday", "year"],
)
show_dataframe(trade_calendar_df.sort_values("calendar_date", ascending=False))

## 2. `dim_futures_variety_calendar`：期货品种日历

In [ ]:
variety_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
variety_calendar_df = lakehouse.read_dataset(
    LAKE_ROOT / "dim_futures_variety_calendar", EXCHANGE_MONTH_PARTITIONING, FUTURES_VARIETY_CALENDAR_SCHEMA,
    ["underlying_code", "exchange_code", "trading_date", "active_contract_count", "year", "month"],
    variety_filter,
)
show_dataframe(variety_calendar_df[variety_calendar_df["underlying_code"] == "CU"].sort_values("trading_date"))

## 3. `dim_futures_contract_calendar`：合约 Session 日历

In [ ]:
contract_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1) & (ds.field("trading_date") == date(2024, 1, 2))
contract_calendar_df = lakehouse.read_dataset(
    LAKE_ROOT / "dim_futures_contract_calendar", EXCHANGE_MONTH_PARTITIONING, FUTURES_CONTRACT_CALENDAR_SCHEMA,
    ["contract_code", "exchange_code", "underlying_code", "trading_date", "session_number", "session_start_at", "session_end_at", "is_night_session", "year", "month"],
    contract_filter,
)
show_dataframe(contract_calendar_df[contract_calendar_df["underlying_code"] == "CU"].sort_values(["contract_code", "session_number"]))

## 4. `dim_futures_session_schedule_signal`：Session 开闭市证据信号

In [ ]:
signal_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
schedule_signal_df = lakehouse.read_dataset(
    LAKE_ROOT / "dim_futures_session_schedule_signal", EXCHANGE_MONTH_PARTITIONING, FUTURES_SESSION_SCHEDULE_SIGNAL_SCHEMA,
    ["exchange_code", "trading_date", "session_text", "session_start_at", "session_end_at", "schedule_status", "signal_reason", "evidence_level", "evidence_source", "is_fetch_exempt", "year", "month"],
    signal_filter,
)
show_dataframe(schedule_signal_df.sort_values(["trading_date", "session_start_at"]))

## 5. `fact_futures_fetch_status`：日线/分钟线拉取状态

In [ ]:
status_filter = (ds.field("bar_frequency") == "1m") & (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
fetch_status_df = lakehouse.read_dataset(
    LAKE_ROOT / "fact_futures_fetch_status", STATUS_PARTITIONING, FUTURES_FETCH_STATUS_SCHEMA,
    ["bar_frequency", "contract_code", "exchange_code", "underlying_code", "trading_date", "session_number", "is_fetch_required", "is_fetch_completed", "is_data_missing", "expected_bar_count", "actual_bar_count", "missing_bar_count", "selection_reason", "year", "month"],
    status_filter,
)
show_dataframe(fetch_status_df[fetch_status_df["underlying_code"] == "CU"].sort_values(["trading_date", "contract_code", "session_number"]))

## 6. `fact_futures_missing_bar`：精确缺失日/分钟

In [ ]:
missing_filter = (ds.field("bar_frequency") == "1d") & (ds.field("exchange_code") == "XSGE") & (ds.field("underlying_code") == "CU") & (ds.field("year") == 2024) & (ds.field("month") == 1)
missing_bar_df = lakehouse.read_dataset(
    LAKE_ROOT / "fact_futures_missing_bar", MISSING_PARTITIONING, FUTURES_MISSING_BAR_SCHEMA,
    ["bar_frequency", "contract_code", "exchange_code", "underlying_code", "trading_date", "session_number", "expected_bar_at", "detected_at", "year", "month"],
    missing_filter,
)
show_dataframe(missing_bar_df.sort_values(["trading_date", "contract_code", "expected_bar_at"]))

## 7. `fact_futures_daily`：期货日线

In [ ]:
daily_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("year") == 2024) & (ds.field("month") == 1)
daily_df = lakehouse.read_dataset(
    LAKE_ROOT / "fact_futures_daily", EXCHANGE_MONTH_PARTITIONING, FUTURES_DAILY_SCHEMA,
    ["contract_code", "exchange_code", "underlying_code", "trading_date", "open", "high", "low", "close", "volume", "open_interest", "has_market_data", "year", "month"],
    daily_filter,
)
show_dataframe(daily_df[(daily_df["underlying_code"] == "CU") & daily_df["has_market_data"]].sort_values(["trading_date", "contract_code"]))

## 8. `fact_futures_minute`：期货分钟线

In [ ]:
minute_filter = (ds.field("exchange_code") == "XSGE") & (ds.field("underlying_code") == "CU") & (ds.field("year") == 2024) & (ds.field("month") == 1) & (ds.field("trading_date") == date(2024, 1, 2))
minute_df = lakehouse.read_dataset(
    LAKE_ROOT / "fact_futures_minute", MINUTE_PARTITIONING, FUTURES_MINUTE_SCHEMA,
    ["contract_code", "exchange_code", "underlying_code", "trading_date", "session_number", "bar_at", "open", "high", "low", "close", "volume", "open_interest", "year", "month"],
    minute_filter,
)
show_dataframe(minute_df.sort_values(["bar_at", "contract_code"]))